# Vector/network maps only (roads + places)
## Imports + parameters

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from datetime import timedelta
from dateutil import parser as dtparser

import geopandas as gpd
from shapely.geometry import Point, LineString
import osmnx as ox

CITY_NAME = "beirut, lebanon"
BUFFER_KM = 0
NETWORK_TYPE = "all"
MIN_VISIT_MIN = 3
MERGE_GAP_MIN = 10
DT_CLIP_MAX_MIN = 30
MAX_SPEED_MPS = 60


# 1) Base polygon (WGS84)
city_gdf = ox.geocode_to_gdf(CITY_NAME)
city_poly = city_gdf.geometry.iloc[0]

# 2) Project to metric CRS and buffer (meters)
utm_crs = city_gdf.estimate_utm_crs()
city_gdf_m = city_gdf.to_crs(utm_crs)
city_poly_m = city_gdf_m.geometry.iloc[0].buffer(BUFFER_KM * 1000)

# 3) Back to WGS84 (this is what OSMnx graph_from_polygon expects)
city_poly = gpd.GeoSeries([city_poly_m], crs=utm_crs).to_crs("EPSG:4326").iloc[0]

# 4) rebuild the network from the NEW buffered polygon
G = ox.graph_from_polygon(city_poly, network_type=NETWORK_TYPE, simplify=True)
G = ox.project_graph(G)  # projects to a metric CRS (often UTM)
edges = ox.graph_to_gdfs(G, nodes=False, fill_edge_geometry=True)

print("Buffered polygon bounds (WGS84):", city_poly.bounds)
print("Edges CRS:", edges.crs)
print("Edges bounds:", edges.total_bounds)



## Helpers

In [ ]:
def parse_geo(s: str):
    s = s.replace("geo:", "")
    lat, lon = s.split(",")
    return float(lat), float(lon)

def duration_minutes(t0, t1):
    return (t1 - t0).total_seconds() / 60.0

def normalize_mode(m):
    if not isinstance(m, str):
        return "unknown"
    m = m.lower()
    if "walk" in m:
        return "walk"
    if "bicy" in m or "cycle" in m:
        return "bike"
    if "bus" in m or "train" in m or "subway" in m or "transit" in m:
        return "transit"
    if "vehicle" in m or "car" in m or "taxi" in m or "motor" in m:
        return "vehicle"
    return m


In [ ]:
import os
import matplotlib.pyplot as plt

def export_plot(fig, filename, dpi=600, folder="outputs", tight=True):
    """
    Export a Matplotlib figure to a high-resolution PNG.

    Parameters
    ----------
    fig : matplotlib.figure.Figure
        The figure object to save.
    filename : str
        File name WITHOUT extension (e.g. "roads_visited").
    dpi : int, optional
        Resolution in dots per inch (default: 600).
    folder : str, optional
        Output directory (default: "outputs").
    tight : bool, optional
        Use tight bounding box (default: True).
    """

    # Ensure output directory exists
    os.makedirs(folder, exist_ok=True)

    path = os.path.join(folder, f"{filename}.png")

    fig.savefig(
        path,
        dpi=dpi,
        bbox_inches="tight" if tight else None,
        pad_inches=0.02,
        facecolor="white"
    )

    print(f"Saved: {path}")


## Load JSON

In [ ]:
path = "location-history.json"
with open(path, "r", encoding="utf-8") as f:
    data = json.load(f)

print(type(data), len(data))


## Extract visits + activities + timelinePath

In [ ]:
rows_visit, rows_act, rows_path = [], [], []

for i, ep in enumerate(data):
    if "startTime" not in ep or "endTime" not in ep:
        continue

    start = dtparser.isoparse(ep["startTime"])
    end   = dtparser.isoparse(ep["endTime"])
    dur = duration_minutes(start, end)
    if dur <= 0:
        continue

    if "visit" in ep:
        v = ep["visit"]
        tc = v.get("topCandidate", {})
        loc = tc.get("placeLocation")
        if loc:
            lat, lon = parse_geo(loc)
            rows_visit.append({
                "episode_id": i,
                "start_time": start,
                "end_time": end,
                "duration_min": dur,
                "lat": lat,
                "lon": lon,
                "place_id": tc.get("placeID"),
                "semantic_type": tc.get("semanticType"),
                "visit_prob": float(v.get("probability", tc.get("probability", 1.0))),
            })

    if "activity" in ep:
        a = ep["activity"]
        if "start" in a and "end" in a:
            slat, slon = parse_geo(a["start"])
            elat, elon = parse_geo(a["end"])
            tc = a.get("topCandidate", {})
            act_prob = a.get("probability", tc.get("probability", 1.0))
            try:
                act_prob = float(act_prob) if act_prob is not None else 1.0
            except (TypeError, ValueError):
                act_prob = 1.0

            rows_act.append({
                "episode_id": i,
                "start_time": start,
                "end_time": end,
                "duration_min": dur,
                "start_lat": slat,
                "start_lon": slon,
                "end_lat": elat,
                "end_lon": elon,
                "distance_m": float(a.get("distanceMeters", 0.0)),
                "mode_raw": tc.get("type"),
                "act_prob": act_prob,
            })

    if "timelinePath" in ep:
        for p in ep["timelinePath"]:
            plat, plon = parse_geo(p["point"])
            offset_min = float(p.get("durationMinutesOffsetFromStartTime", 0.0))
            t = start + timedelta(minutes=offset_min)
            rows_path.append({
                "episode_id": i,
                "time": t,
                "lat": plat,
                "lon": plon,
            })

visits = pd.DataFrame(rows_visit)
acts   = pd.DataFrame(rows_act)
pathpts= pd.DataFrame(rows_path)

print("visits:", len(visits), "acts:", len(acts), "pathpts:", len(pathpts))


## Parse times + clean + weights

In [ ]:
# times
visits["start_time"] = pd.to_datetime(visits["start_time"], utc=True)
visits["end_time"]   = pd.to_datetime(visits["end_time"],   utc=True)
pathpts["time"]      = pd.to_datetime(pathpts["time"], utc=True)

if len(acts) > 0:
    acts["start_time"] = pd.to_datetime(acts["start_time"], utc=True)
    acts["end_time"]   = pd.to_datetime(acts["end_time"],   utc=True)

# visits: filter + weight
visits = visits.dropna(subset=["lat","lon","duration_min"])
visits = visits[visits["duration_min"] >= MIN_VISIT_MIN].copy()
visits["weight"] = visits["duration_min"] * visits["visit_prob"]

# merge same place with short gaps
visits = visits.sort_values(["start_time"])
visits["prev_place"] = visits["place_id"].shift(1)
visits["prev_end"]   = visits["end_time"].shift(1)
gap = (visits["start_time"] - visits["prev_end"]).dt.total_seconds()/60

merge_mask = (visits["place_id"].notna()) & (visits["place_id"] == visits["prev_place"]) & (gap <= MERGE_GAP_MIN)
visits["group"] = (~merge_mask).cumsum()

visits = (visits.groupby("group", as_index=False)
          .agg({
              "episode_id":"first",
              "start_time":"min",
              "end_time":"max",
              "duration_min":"sum",
              "lat":"first","lon":"first",
              "place_id":"first",
              "semantic_type":"first",
              "visit_prob":"mean",
              "weight":"sum"
          }))

# clean activities (for mode labeling, QA)
if len(acts) > 0:
    acts = acts.dropna(subset=["start_lat","start_lon","end_lat","end_lon","duration_min"]).copy()
    acts = acts[acts["duration_min"] > 0]
    acts["speed_mps"] = acts["distance_m"] / (acts["duration_min"]*60)
    acts = acts[acts["speed_mps"] < MAX_SPEED_MPS]
    acts["mode"] = acts["mode_raw"].apply(normalize_mode)

print("visits merged:", len(visits))


## City polygon + GeoDataFrames

In [ ]:


gvis = gpd.GeoDataFrame(
    visits,
    geometry=[Point(xy) for xy in zip(visits["lon"], visits["lat"])],
    crs="EPSG:4326"
)

gpath = gpd.GeoDataFrame(
    pathpts,
    geometry=[Point(xy) for xy in zip(pathpts["lon"], pathpts["lat"])],
    crs="EPSG:4326"
)


## Build road network + edges GeoDataFrame

In [ ]:
# G = ox.graph_from_polygon(city_poly, network_type=NETWORK_TYPE, simplify=True)
# G = ox.project_graph(G)

# edges = ox.graph_to_gdfs(G, nodes=False, fill_edge_geometry=True)
# edges = edges.reset_index(drop=False)  # ensures u,v,key columns exist in all versions
# print("edges:", len(edges), "CRS:", edges.crs)


## Project points to graph CRS + filter within city

In [ ]:
# city_m = city_gdf.to_crs(edges.crs)
# city_poly_m = city_m.geometry.iloc[0]
# Project the buffered polygon into edges CRS for filtering
city_poly_m = gpd.GeoSeries([city_poly], crs="EPSG:4326").to_crs(edges.crs).iloc[0]



gvis_p  = gvis.to_crs(edges.crs)
gpath_p = gpath.to_crs(edges.crs)

gvis_p  = gvis_p[gvis_p.within(city_poly_m)]
gpath_p = gpath_p[gpath_p.within(city_poly_m)]

# clip to graph bounds (prevents snapping to nonsense far edges)
minx, miny, maxx, maxy = edges.total_bounds
gvis_p  = gvis_p.cx[minx:maxx, miny:maxy]
gpath_p = gpath_p.cx[minx:maxx, miny:maxy]

print("gvis_p:", len(gvis_p), "gpath_p:", len(gpath_p))


## Compute movement weights on timelinePath (Δt)

In [ ]:
gpath_p = gpath_p.sort_values(["episode_id", "time"])
gpath_p["t_next"] = gpath_p.groupby("episode_id")["time"].shift(-1)

gpath_p["dt_min"] = (gpath_p["t_next"] - gpath_p["time"]).dt.total_seconds()/60.0
gpath_p["dt_min"] = gpath_p["dt_min"].fillna(0).clip(lower=0, upper=DT_CLIP_MAX_MIN)

gpath_p = gpath_p[gpath_p["dt_min"] > 0].copy()
gpath_p["weight"] = gpath_p["dt_min"]


## Snap points to nearest edges (robust across OSMnx versions)

In [ ]:
def snap_points_to_edges(G, gdf_points):
    uvk = ox.distance.nearest_edges(
        G,
        X=gdf_points.geometry.x.values,
        Y=gdf_points.geometry.y.values
    )

    uvk_arr = np.asarray(uvk)
    if uvk_arr.ndim == 2 and uvk_arr.shape[1] == 1:
        uvk_arr = uvk_arr[:, 0]

    def unpack_edge(e):
        if isinstance(e, tuple):
            if len(e) == 3:
                return e
            if len(e) == 2:
                u, v = e
                return (u, v, 0)
        return (np.nan, np.nan, np.nan)

    uvk_unpacked = np.array([unpack_edge(e) for e in uvk_arr], dtype=object)

    out = gdf_points.copy()
    out["u"] = uvk_unpacked[:, 0]
    out["v"] = uvk_unpacked[:, 1]
    out["key"] = uvk_unpacked[:, 2]
    out = out.dropna(subset=["u","v","key"])
    return out

gpath_s = snap_points_to_edges(G, gpath_p)
print("snapped points:", len(gpath_s))
gpath_s[["u","v","key","weight"]].head()


## Aggregate edge weights + plot roads

In [ ]:
edge_move = (
    gpath_s.groupby(["u","v","key"])["weight"]
    .sum()
    .rename("w_move_min")
    .reset_index()
)

edges_w = edges.merge(edge_move, on=["u","v","key"], how="left")
edges_w["w_move_min"] = pd.to_numeric(edges_w["w_move_min"], errors="coerce").fillna(0.0).clip(lower=0.0)

edges_w["w_plot"] = np.log1p(edges_w["w_move_min"])
if edges_w["w_plot"].gt(0).any():
    q99 = edges_w.loc[edges_w["w_plot"] > 0, "w_plot"].quantile(0.99)
    edges_w["w_plot"] = edges_w["w_plot"].clip(upper=q99)

fig, ax = plt.subplots(figsize=(12, 12))
ax.set_axis_off()

# base roads
edges.plot(ax=ax, linewidth=0.6, alpha=0.20)

hot = edges_w[edges_w["w_move_min"] > 0]
if len(hot) > 0:
    lw = 0.5 + 4.0 * (hot["w_plot"] / hot["w_plot"].max())
    hot.plot(ax=ax, linewidth=lw, column="w_plot", legend=True)

ax.set_title("Roads I frequent (time-weighted from timelinePath)")
export_plot(fig, "frequented_roads_general")
plt.close()


## Places map (visit dwell)

In [ ]:
places = gvis_p.copy()
places["w_place"] = pd.to_numeric(places["weight"], errors="coerce").fillna(0.0).clip(lower=0.0)

places["w_plot"] = np.log1p(places["w_place"])
if places["w_plot"].gt(0).any():
    q99p = places.loc[places["w_plot"] > 0, "w_plot"].quantile(0.99)
    places["w_plot"] = places["w_plot"].clip(upper=q99p)

fig, ax = plt.subplots(figsize=(12, 12))
ax.set_axis_off()
edges.plot(ax=ax, linewidth=0.6, alpha=0.20)

s = 10 + 150 * (places["w_plot"] / places["w_plot"].max() if places["w_plot"].max() > 0 else 0)

places.plot(
    ax=ax,
    markersize=s,
    column="w_plot",
    legend=True,
    alpha=0.85
)

ax.set_title("Places I frequent (dwell-time weighted)")
plt.savefig("outputs/places_frequented.png", dpi=300, bbox_inches="tight")
plt.close()
print("Saved outputs/places_frequented.png")


## POI labeling (nearest OSM features)

In [ ]:
# --- POI labeling (nearest OSM features) ---

pois = ox.features_from_polygon(
    city_poly,
    tags={"amenity": True, "shop": True, "tourism": True}
).to_crs(edges.crs)

# Keep valid geometries only
pois = pois[pois.geometry.notnull() & pois.is_valid].copy()

# Turn polygons into centroids for nearest-join
geom_type = pois.geometry.geom_type
pois.loc[geom_type.isin(["Polygon", "MultiPolygon"]), "geometry"] = pois.loc[
    geom_type.isin(["Polygon", "MultiPolygon"]), "geometry"
].centroid

pois = gpd.GeoDataFrame(pois, geometry="geometry", crs=edges.crs)

joined = gpd.sjoin_nearest(
    places[["w_place", "geometry"]],
    pois[["name", "amenity", "shop", "tourism", "geometry"]],
    how="left",
    distance_col="dist_m"
)


MAX_MATCH_DIST_M = 150
joined = joined[joined["dist_m"].notna() & (joined["dist_m"] <= MAX_MATCH_DIST_M)].copy()


joined["poi_type"] = (
    joined["amenity"].astype("string")
    .fillna(joined["shop"].astype("string"))
    .fillna(joined["tourism"].astype("string"))
    .fillna("unknown")
)


joined["poi_label"] = joined["name"].astype("string").fillna(joined["poi_type"])

poi_scores_named = (
    joined.groupby("poi_label")["w_place"]
    .sum()
    .sort_values(ascending=False)
)

poi_scores_types = (
    joined.groupby("poi_type")["w_place"]
    .sum()
    .sort_values(ascending=False)
)

tmp = joined.copy()
for c in ["name", "amenity", "shop", "tourism"]:
    tmp[c] = tmp[c].astype("string").fillna("")

poi_scores_strict = (
    tmp.groupby(["name", "amenity", "shop", "tourism"])["w_place"]
    .sum()
    .sort_values(ascending=False)
)

print("Top named/labelled POIs:")
poi_scores_named.head(50).to_csv("outputs/poi_scores_named.csv")
print("Saved outputs/poi_scores_named.csv")

print("Top POI categories:")
poi_scores_types.head(50).to_csv("outputs/poi_scores_types.csv")
print("Saved outputs/poi_scores_types.csv")

print("Top strict (name+amenity+shop+tourism):")
poi_scores_strict.head(50).to_csv("outputs/poi_scores_strict.csv")
print("Saved outputs/poi_scores_strict.csv")


In [ ]:
print("POIs:", len(pois))
print("Places:", len(places))
print("Joined rows:", len(joined))

print("POIs with names:", pois["name"].notna().mean())
print("Joined with names:", joined["name"].notna().mean())


## Weekdays vs weekends road maps

In [ ]:
gpath_s2 = gpath_s.copy()
gpath_s2["weekday"] = gpath_s2["time"].dt.weekday
gpath_s2["is_weekend"] = gpath_s2["weekday"] >= 5

for label, sub in [
    ("Weekdays", gpath_s2[~gpath_s2["is_weekend"]]),
    ("Weekends", gpath_s2[gpath_s2["is_weekend"]]),
]:
    edge_move = (
        sub.groupby(["u","v","key"])["weight"]
        .sum()
        .rename("w")
        .reset_index()
    )

    edges_tmp = edges.merge(edge_move, on=["u","v","key"], how="left")
    edges_tmp["w"] = pd.to_numeric(edges_tmp["w"], errors="coerce").fillna(0.0).clip(lower=0.0)

    edges_tmp["w_plot"] = np.log1p(edges_tmp["w"])
    if edges_tmp["w_plot"].gt(0).any():
        q99 = edges_tmp.loc[edges_tmp["w_plot"] > 0, "w_plot"].quantile(0.99)
        edges_tmp["w_plot"] = edges_tmp["w_plot"].clip(upper=q99)

    fig, ax = plt.subplots(figsize=(12, 12))
    ax.set_axis_off()
    edges.plot(ax=ax, linewidth=0.6, alpha=0.20)

    hot = edges_tmp[edges_tmp["w"] > 0]
    if len(hot) > 0:
        lw = 0.5 + 4.0 * (hot["w_plot"] / hot["w_plot"].max())
        hot.plot(ax=ax, linewidth=lw, column="w_plot", legend=False)

    ax.set_title(f"Roads I frequent — {label}")
    safe_label = label.replace(" ", "_").lower()
    plt.savefig(f"outputs/roads_frequent_{safe_label}.png", dpi=300, bbox_inches="tight")
    plt.close()
    print(f"Saved outputs/roads_frequent_{safe_label}.png")


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# --- parameters for POI match quality ---
MAX_MATCH_DIST_M = 150  # tighten to 75-100 for higher precision

# joined was created from sjoin_nearest(places -> pois)
places_with_poi = joined.copy()

# keep only plausible matches
places_with_poi = places_with_poi[places_with_poi["dist_m"].notna() & (places_with_poi["dist_m"] <= MAX_MATCH_DIST_M)].copy()

# keep the same geometry as places (left geometry), already in joined
places_with_poi = places_with_poi.set_geometry("geometry")
places_with_poi = places_with_poi[places_with_poi["w_place"].notna()].copy()

# log scale for plotting (not for meaning)
places_with_poi["w_plot"] = np.log1p(places_with_poi["w_place"].clip(lower=0))

print("places_with_poi rows:", len(places_with_poi))
print("Top categories:", places_with_poi["poi_type"].value_counts().head(10))


In [ ]:
from matplotlib.lines import Line2D
import matplotlib.cm as cm
import matplotlib.colors as mcolors

# pick categories you care about (top N)
top_types = (
    places_with_poi.groupby("poi_type")["w_place"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .index.tolist()
)

plot_df = places_with_poi[places_with_poi["poi_type"].isin(top_types)].copy()

fig, ax = plt.subplots(figsize=(12, 12))
ax.set_axis_off()

# base roads
edges.plot(ax=ax, linewidth=0.6, alpha=0.20)

# ---------- COLOR CONTROL  ----------

# stable categorical colormap
cmap = cm.get_cmap("tab10", len(top_types))

# category → color mapping
cat_color = {
    cat: mcolors.to_hex(cmap(i))
    for i, cat in enumerate(top_types)
}

# ---------- SIZE SCALING ----------

sizes = 8 + 160 * (
    plot_df["w_plot"] / plot_df["w_plot"].max()
    if plot_df["w_plot"].max() > 0 else 0
)

# ---------- PLOT POINTS ----------

for cat in top_types:
    sub = plot_df[plot_df["poi_type"] == cat]
    if len(sub) == 0:
        continue

    sub.plot(
        ax=ax,
        color=cat_color[cat],
        markersize=sizes.loc[sub.index],
        alpha=0.85
    )

# ---------- LEGEND ----------

legend_elements = [
    Line2D(
        [0], [0],
        marker='o',
        linestyle='',
        markersize=8,
        markerfacecolor=cat_color[cat],
        markeredgecolor='black',
        label=cat
    )
    for cat in top_types
]

ax.legend(
    handles=legend_elements,
    title="Visit purpose (nearest OSM category)",
    loc="upper right",
    frameon=True
)

ax.set_title("Places I frequent — colored by visit purpose (time-weighted)")
plt.savefig("outputs/places_colored_by_purpose.png", dpi=300, bbox_inches="tight")
plt.close()
print("Saved outputs/places_colored_by_purpose.png")


In [ ]:
top_types = (places_with_poi.groupby("poi_type")["w_place"].sum()
             .sort_values(ascending=False)
             .head(8)  
             .index.tolist())

for cat in top_types:
    sub = places_with_poi[places_with_poi["poi_type"] == cat].copy()
    if len(sub) == 0:
        continue

    sub["w_plot"] = np.log1p(sub["w_place"].clip(lower=0))
    sizes = 10 + 180 * (sub["w_plot"] / sub["w_plot"].max() if sub["w_plot"].max() > 0 else 0)

    fig, ax = plt.subplots(figsize=(12, 12))
    ax.set_axis_off()
    edges.plot(ax=ax, linewidth=0.6, alpha=0.20)

    sub.plot(ax=ax, markersize=sizes, alpha=0.9)

    ax.set_title(f"Places I frequent — {cat} (time-weighted)")
    safe_cat = cat.replace(" ", "_").lower().replace("/", "_")
    plt.savefig(f"outputs/places_cat_{safe_cat}.png", dpi=300, bbox_inches="tight")
    plt.close()
    print(f"Saved outputs/places_cat_{safe_cat}.png")


In [ ]:
# Choose top N POI labels by total weighted time
poi_rank = (places_with_poi.groupby(["poi_label", "poi_type"])["w_place"]
            .sum()
            .sort_values(ascending=False)
            .reset_index())

TOP_N = 15
top_pois = poi_rank.head(TOP_N)


centers = (places_with_poi[places_with_poi["poi_label"].isin(top_pois["poi_label"])]
           .groupby("poi_label")["geometry"]
           .apply(lambda s: s.unary_union.centroid))

fig, ax = plt.subplots(figsize=(12, 12))
ax.set_axis_off()
edges.plot(ax=ax, linewidth=0.6, alpha=0.20)

# plot the points contributing to these POIs
sub = places_with_poi[places_with_poi["poi_label"].isin(top_pois["poi_label"])].copy()
sub["w_plot"] = np.log1p(sub["w_place"].clip(lower=0))
sizes = 15 + 240 * (sub["w_plot"] / sub["w_plot"].max() if sub["w_plot"].max() > 0 else 0)
sub.plot(ax=ax, markersize=sizes, alpha=0.75)

# annotate centroid labels
for label, geom in centers.items():
    x, y = geom.x, geom.y
    ax.text(x, y, label, fontsize=9)

ax.set_title(f"Top {TOP_N} named/labelled POIs (time-weighted)")
plt.savefig("outputs/top_named_pois_map.png", dpi=300, bbox_inches="tight")
plt.close()
print("Saved outputs/top_named_pois_map.png")


In [ ]:
# Choose categories that matter (top K by total time)
top_types = (places_with_poi.groupby("poi_type")["w_place"].sum()
             .sort_values(ascending=False)
             .head(6)
             .index.tolist())

# hexbin settings
GRIDSIZE = 65  # higher = finer detail
MINCNT = 1     # only show bins with data

fig, axes = plt.subplots(2, 3, figsize=(18, 11))
axes = axes.ravel()

for ax, cat in zip(axes, top_types):
    ax.set_axis_off()
    edges.plot(ax=ax, linewidth=0.5, alpha=0.18)

    sub = places_with_poi[places_with_poi["poi_type"] == cat]
    if len(sub) == 0:
        ax.set_title(f"{cat} (no data)")
        continue

    x = sub.geometry.x.values
    y = sub.geometry.y.values
    w = sub["w_place"].values  # time-weighted dwell

    hb = ax.hexbin(
        x, y,
        C=w,
        reduce_C_function=np.sum,
        gridsize=GRIDSIZE,
        mincnt=MINCNT,
        alpha=0.85
    )
    ax.set_title(f"{cat} — heat (sum of weighted minutes)")

# shared colorbar
cbar = fig.colorbar(hb, ax=axes.tolist(), fraction=0.02, pad=0.01)
cbar.set_label("Sum of weighted minutes in hex cell")

fig.suptitle("Visited places heatmaps by purpose (hexbin, time-weighted)", y=0.98)
plt.savefig("outputs/places_heatmaps_by_purpose.png", dpi=300, bbox_inches="tight")
plt.close()
print("Saved outputs/places_heatmaps_by_purpose.png")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Choose the point:
# - gpath_p: timelinePath points
# - gvis_p: visit points
# - places_with_poi: filtered visit points
points = gpath_p.copy()


# points["weight"] = points["dt_min"].clip(lower=0, upper=30)
# Otherwise, use 1 per point:
if "weight" not in points.columns:
    points["weight"] = 1.0

# Plot extents from roads (keeps map framed nicely)
minx, miny, maxx, maxy = edges.total_bounds

fig, ax = plt.subplots(figsize=(12, 12))
ax.set_axis_off()

# Base vector roads
edges.plot(ax=ax, linewidth=0.6, alpha=0.20)

# Hexbin intensity
x = points.geometry.x.values
y = points.geometry.y.values
w = points["weight"].values

hb = ax.hexbin(
    x, y,
    C=w,
    reduce_C_function=np.sum,   # intensity = sum of weights in each hex
    gridsize=150,                # higher => finer detail
    mincnt=1,
    alpha=0.75
)

ax.set_xlim(minx, maxx)
ax.set_ylim(miny, maxy)

cbar = fig.colorbar(hb, ax=ax, fraction=0.03, pad=0.01)
cbar.set_label("Point intensity (sum of weights per hex cell)")

ax.set_title("Point-intensity field over roads (all points)")
plt.savefig("outputs/hexbin_all_points.png", dpi=300, bbox_inches="tight")
plt.close()
print("Saved outputs/hexbin_all_points.png")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from sklearn.neighbors import KernelDensity

points = gpath_p.copy()
if "weight" not in points.columns:
    points["weight"] = 1.0

# Bounds
minx, miny, maxx, maxy = edges.total_bounds

CELL = 10   # meters per pixel
BW = 50     # KDE bandwidth (meters)

xs = np.arange(minx, maxx, CELL)
ys = np.arange(miny, maxy, CELL)
xx, yy = np.meshgrid(xs, ys)
grid = np.vstack([xx.ravel(), yy.ravel()]).T

coords = np.vstack([points.geometry.x.values, points.geometry.y.values]).T
weights = points["weight"].values

kde = KernelDensity(bandwidth=BW, kernel="gaussian")
kde.fit(coords, sample_weight=weights)

dens = np.exp(kde.score_samples(grid)).reshape(xx.shape)

# ---- TRANSPARENCY FIX ----
eps = dens.max() * 0.0001          # hide background noise
dens_masked = np.ma.masked_less(dens, eps)

# ---- COLORMAP FIX ----
cmap = plt.cm.autumn_r.copy()    
cmap.set_bad(alpha=0.0)

norm = mcolors.LogNorm(vmin=eps, vmax=dens.max())

# ---- PLOT ----
fig, ax = plt.subplots(figsize=(12, 12))
ax.set_axis_off()

im = ax.imshow(
    dens_masked,
    extent=[minx, maxx, miny, maxy],
    origin="lower",
    cmap=cmap,
    norm=norm
)

edges.plot(ax=ax, linewidth=0.6, alpha=0.25)

cbar = fig.colorbar(im, ax=ax, fraction=0.03, pad=0.01)
cbar.set_label("Point intensity (time-weighted, log scale)")

ax.set_title("Point-intensity field over roads (where I actually spent time)")
plt.savefig("outputs/kde_all_points.png", dpi=300, bbox_inches="tight")
plt.close()
print("Saved outputs/kde_all_points.png")


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import osmnx as ox

if not {"u","v","key"}.issubset(edges.columns):
    edges = edges.reset_index(drop=False)

pts = gpath_p.copy()
if "weight" not in pts.columns:
    pts["weight"] = 1.0

uvk = ox.distance.nearest_edges(G, X=pts.geometry.x.values, Y=pts.geometry.y.values)

uvk_arr = np.asarray(uvk)
if uvk_arr.ndim == 2 and uvk_arr.shape[1] == 1:
    uvk_arr = uvk_arr[:, 0]

def unpack_edge(e):
    if isinstance(e, tuple):
        if len(e) == 3:
            return e
        if len(e) == 2:
            u, v = e
            return (u, v, 0)
    return (np.nan, np.nan, np.nan)

uvk_unpacked = np.array([unpack_edge(e) for e in uvk_arr], dtype=object)

pts["u"] = uvk_unpacked[:, 0]
pts["v"] = uvk_unpacked[:, 1]
pts["key"] = uvk_unpacked[:, 2]
pts = pts.dropna(subset=["u","v","key"])

visited_edges = pts.groupby(["u","v","key"])["weight"].sum().rename("w").reset_index()
visited_keys = set(map(tuple, visited_edges[["u","v","key"]].values))

edges2 = edges.copy()
edges2["visited"] = list(map(tuple, edges2[["u","v","key"]].values))
edges2["visited"] = edges2["visited"].isin(visited_keys)

fig, ax = plt.subplots(figsize=(12, 12))
ax.set_axis_off()

edges2[~edges2["visited"]].plot(ax=ax, color="red", linewidth=0.6, alpha=0.55)
edges2[edges2["visited"]].plot(ax=ax, color="black", linewidth=0.8, alpha=0.85)

ax.set_title("Road network coverage: black = visited, red = never visited (within downloaded region)")
export_plot(fig, "unfrequented_roads")
plt.show()

print("Total edges:", len(edges2))
print("Visited edges:", int(edges2['visited'].sum()))
print("Never visited edges:", int((~edges2['visited']).sum()))
print("Visited %:", round(100 * edges2['visited'].mean(), 2))


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from matplotlib.lines import Line2D
import osmnx as ox

# -------------------------
# CONFIG (edit these)
# -------------------------
MODE_PRIORITY = ["walk", "bike", "transit", "vehicle", "unknown"]  # left = higher priority
BASE_ROAD_ALPHA = 0.15
MODE_ROAD_ALPHA = 0.90
BASE_ROAD_WIDTH = 0.50
MODE_ROAD_WIDTH = 1.20

# -------------------------
# 0) Ensure edges has u,v,key columns
# -------------------------
if not {"u","v","key"}.issubset(edges.columns):
    edges = edges.reset_index(drop=False)

# -------------------------
# 1) Snap all timeline points to nearest edges
# -------------------------
pts = gpath_p.copy()

uvk = ox.distance.nearest_edges(
    G,
    X=pts.geometry.x.values,
    Y=pts.geometry.y.values
)

uvk_arr = np.asarray(uvk)
if uvk_arr.ndim == 2 and uvk_arr.shape[1] == 1:
    uvk_arr = uvk_arr[:, 0]

def unpack_edge(e):
    if isinstance(e, tuple):
        if len(e) == 3:
            return e
        if len(e) == 2:
            u, v = e
            return (u, v, 0)
    return (np.nan, np.nan, np.nan)

uvk_unpacked = np.array([unpack_edge(e) for e in uvk_arr], dtype=object)

pts["u"] = uvk_unpacked[:, 0]
pts["v"] = uvk_unpacked[:, 1]
pts["key"] = uvk_unpacked[:, 2]
pts = pts.dropna(subset=["u","v","key"])

# -------------------------
# 2) Attach mode to each timeline point via episode_id -> acts.mode
# -------------------------
episode_to_mode = acts.set_index("episode_id")["mode"] if "mode" in acts.columns else pd.Series(dtype="object")
pts["mode"] = pts["episode_id"].map(episode_to_mode).fillna("unknown")

# Optional: keep only known modes (comment out if you want everything)
# pts = pts[pts["mode"].isin(MODE_PRIORITY)].copy()

# -------------------------
# 3) Create a visited-by-mode table (binary, no frequency)
# -------------------------
# One row per unique edge+mode that occurred at least once
edge_mode = pts[["u","v","key","mode"]].drop_duplicates()

# Build a per-edge list of modes used
modes_per_edge = (
    edge_mode.groupby(["u","v","key"])["mode"]
    .apply(lambda s: sorted(set(s)))
    .rename("modes_used")
    .reset_index()
)

# Assign a single display mode by priority (handles multi-mode edges)
priority_rank = {m: i for i, m in enumerate(MODE_PRIORITY)}

def choose_mode(modes):
    # pick the highest-priority mode present
    modes = [m for m in modes if m in priority_rank]
    if not modes:
        return "unknown"
    return sorted(modes, key=lambda m: priority_rank[m])[0]

modes_per_edge["mode_primary"] = modes_per_edge["modes_used"].apply(choose_mode)

# Join back to edges
edges_m = edges.merge(modes_per_edge, on=["u","v","key"], how="left")
edges_m["mode_primary"] = edges_m["mode_primary"].fillna("unvisited")

# -------------------------
# 4) Colors (categorical)
# -------------------------
modes_to_plot = [m for m in MODE_PRIORITY if m != "unknown"] + ["unknown"]
# include only modes that actually appear
present = [m for m in modes_to_plot if (edges_m["mode_primary"] == m).any()]
# keep unvisited separate (optional)
present_unvisited = (edges_m["mode_primary"] == "unvisited").any()

cmap = cm.get_cmap("tab10", max(3, len(present)))
mode_color = {m: mcolors.to_hex(cmap(i)) for i, m in enumerate(present)}
mode_color["unvisited"] = "#cccccc"  # light gray

# -------------------------
# 5) Plot
# -------------------------
fig, ax = plt.subplots(figsize=(12, 12))
ax.set_axis_off()

# base network (very light)
edges.plot(ax=ax, linewidth=BASE_ROAD_WIDTH, alpha=BASE_ROAD_ALPHA, color="black")

# plot visited by mode
for m in present:
    sub = edges_m[edges_m["mode_primary"] == m]
    sub.plot(ax=ax, linewidth=MODE_ROAD_WIDTH, alpha=MODE_ROAD_ALPHA, color=mode_color[m])

# optional: show unvisited roads lightly (comment out if you only want visited)
if present_unvisited:
    edges_m[edges_m["mode_primary"] == "unvisited"].plot(
        ax=ax, linewidth=0.45, alpha=0.12, color=mode_color["unvisited"]
    )

# legend
legend_items = [
    Line2D([0],[0], color=mode_color[m], lw=3, label=m)
    for m in present
]
if present_unvisited:
    legend_items.append(Line2D([0],[0], color=mode_color["unvisited"], lw=3, label="unvisited"))

ax.legend(handles=legend_items, title="Mode (binary: visited at least once)", loc="upper right", frameon=True)

ax.set_title("Roads visited — separated by mode of transport (binary coverage, no frequency)")
export_plot(fig, "roads_visited_by_mode_binary")
plt.show()


# quick counts
print(edges_m["mode_primary"].value_counts().head(20))


In [ ]:
# import os
# import numpy as np
# import matplotlib.pyplot as plt
# from matplotlib.animation import FuncAnimation, FFMpegWriter, PillowWriter

# def make_trail_animation(
#     points_gdf,
#     edges_gdf=None,
#     out_path="outputs/movement_trail.mp4",
#     window_points=500,
#     step=5,
#     fps=30,
#     dpi=150,
#     figsize=(10, 10),
#     base_edge_alpha=0.18,
#     base_edge_lw=0.5,
#     trail_lw=2.0,
#     trail_alpha=0.95,
#     trail_color="black",
#     facecolor="white",
#     max_frames=None
# ):
#     """
#     Create a 'sliding-window' movement animation:
#       - first N points draw a polyline
#       - then the window moves forward: at i+1, drop point i-(N-1)
#       - persists for `window_points` points

#     Parameters
#     ----------
#     points_gdf : GeoDataFrame
#         Must have Point geometry and a time column named 'time' (datetime).
#         CRS should match edges_gdf if provided.
#     edges_gdf : GeoDataFrame or None
#         Road network edges to plot as a faint basemap (optional).
#     out_path : str
#         Output video path (.mp4) or gif (.gif). Folder is created if needed.
#     window_points : int
#         Number of points kept in the trail at any frame.
#     step : int
#         Advance this many points per frame (larger = faster render).
#     fps : int
#         Frames per second for the output video/gif.
#     dpi : int
#         Output resolution (higher = sharper, slower).
#     max_frames : int or None
#         Cap frames to limit runtime (None = full length).

#     Notes
#     -----
#     - For MP4 output, you need ffmpeg installed on your system.
#     - For GIF output, set out_path ending with .gif (slower, larger files).
#     """

#     if "time" not in points_gdf.columns:
#         raise ValueError("points_gdf must contain a 'time' column (datetime).")

#     # Ensure output directory exists
#     os.makedirs(os.path.dirname(out_path) or ".", exist_ok=True)

#     # Sort by time and drop missing geometries
#     pts = points_gdf.dropna(subset=["geometry"]).sort_values("time").copy()
#     if len(pts) < 2:
#         raise ValueError("Need at least 2 points to animate a path.")

#     x = pts.geometry.x.to_numpy()
#     y = pts.geometry.y.to_numpy()

#     # Set bounds (prefer edges bounds if provided)
#     if edges_gdf is not None and len(edges_gdf) > 0:
#         minx, miny, maxx, maxy = edges_gdf.total_bounds
#     else:
#         minx, maxx = np.nanmin(x), np.nanmax(x)
#         miny, maxy = np.nanmin(y), np.nanmax(y)

#     # Add a bit of padding
#     pad_x = (maxx - minx) * 0.03 if maxx > minx else 1.0
#     pad_y = (maxy - miny) * 0.03 if maxy > miny else 1.0

#     fig, ax = plt.subplots(figsize=figsize)
#     fig.patch.set_facecolor(facecolor)
#     ax.set_facecolor(facecolor)
#     ax.set_axis_off()
#     ax.set_xlim(minx - pad_x, maxx + pad_x)
#     ax.set_ylim(miny - pad_y, maxy + pad_y)

#     # Base map
#     if edges_gdf is not None and len(edges_gdf) > 0:
#         edges_gdf.plot(ax=ax, linewidth=base_edge_lw, alpha=base_edge_alpha)

#     # Trail line + head point
#     (trail_line,) = ax.plot([], [], lw=trail_lw, alpha=trail_alpha, color=trail_color)
#     (head_dot,) = ax.plot([], [], marker="o", markersize=4, alpha=0.9, color=trail_color)

#     # Text timestamp
#     time_text = ax.text(
#         0.01, 0.99, "", transform=ax.transAxes,
#         ha="left", va="top", fontsize=10, color="black"
#     )

#     n = len(x)
#     indices = np.arange(0, n, step, dtype=int)

#     if max_frames is not None:
#         indices = indices[:max_frames]

#     def init():
#         trail_line.set_data([], [])
#         head_dot.set_data([], [])
#         time_text.set_text("")
#         return trail_line, head_dot, time_text

#     def update(frame_idx):
#         i = indices[frame_idx]
#         start = max(0, i - window_points + 1)

#         xs = x[start:i+1]
#         ys = y[start:i+1]

#         trail_line.set_data(xs, ys)
#         head_dot.set_data([x[i]], [y[i]])

#         t = pts["time"].iloc[i]
#         # readable timestamp (localize/format as you like)
#         time_text.set_text(t.strftime("%Y-%m-%d %H:%M:%S"))

#         return trail_line, head_dot, time_text

#     anim = FuncAnimation(
#         fig, update,
#         frames=len(indices),
#         init_func=init,
#         blit=True,
#         interval=1000 / fps
#     )

#     # Save
#     ext = os.path.splitext(out_path)[1].lower()
#     if ext == ".gif":
#         writer = PillowWriter(fps=fps)
#         anim.save(out_path, writer=writer, dpi=dpi)
#     else:
#         # MP4 by default
#         writer = FFMpegWriter(fps=fps, bitrate=1800)
#         anim.save(out_path, writer=writer, dpi=dpi)

#     plt.close(fig)
#     print(f"Saved animation: {out_path}")


# # -------------------------
# # HOW TO USE (example)
# # -------------------------
# # 1) Use timelinePath points in the SAME CRS as your edges map:
# #    - gpath_p is your projected timeline points GeoDataFrame (with 'time')
# #    - edges is your road GeoDataFrame (same CRS)

# # Example:
# make_trail_animation(
#     points_gdf=gpath_p,
#     edges_gdf=edges,
#     out_path="outputs/paris_trail.mp4",
#     window_points=100,
#     step=5,     # smaller = smoother but slower (try 1–10)
#     fps=30,
#     dpi=200
# )

# # If you want GIF instead:
# # make_trail_animation(gpath_p, edges, out_path="outputs/paris_trail.gif", window_points=500, step=5, fps=20, dpi=120)


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import osmnx as ox
import networkx as nx
from shapely.geometry import LineString
from matplotlib.animation import FuncAnimation, FFMpegWriter, PillowWriter

# -----------------------------
# 1) Map-match helper: points -> route polyline through roads
# -----------------------------
def build_matched_route(
    G,
    points_gdf,
    time_col="time",
    step=5,
    max_snap_dist_m=80,
    weight="length",
):
    pts = points_gdf.dropna(subset=["geometry"]).copy()
    pts = pts.sort_values(time_col)
    pts = pts.iloc[::step].copy()

    if len(pts) < 2:
        raise ValueError("Not enough points after subsampling.")

    X = pts.geometry.x.values
    Y = pts.geometry.y.values
    times = pd.to_datetime(pts[time_col]).values

    node_ids = ox.distance.nearest_nodes(G, X=X, Y=Y)

    nodes_gdf = ox.graph_to_gdfs(G, nodes=True, edges=False)
    snapped_xy = nodes_gdf.loc[node_ids][["x", "y"]].to_numpy()
    d = np.sqrt((X - snapped_xy[:, 0])**2 + (Y - snapped_xy[:, 1])**2)

    keep = d <= max_snap_dist_m
    node_ids = np.array(node_ids)[keep]
    times = times[keep]

    if len(node_ids) < 2:
        raise ValueError("Too few points after snap filtering.")

    route_x, route_y, route_t = [], [], []
    n_segments, n_no_path = 0, 0

    for u, v, t in zip(node_ids[:-1], node_ids[1:], times[1:]):
        if u == v:
            continue
        try:
            path = nx.shortest_path(G, u, v, weight=weight)
        except (nx.NetworkXNoPath, nx.NodeNotFound):
            n_no_path += 1
            continue

        xs = nodes_gdf.loc[path]["x"].to_numpy()
        ys = nodes_gdf.loc[path]["y"].to_numpy()

        if len(route_x) > 0:
            xs = xs[1:]
            ys = ys[1:]

        route_x.extend(xs.tolist())
        route_y.extend(ys.tolist())
        route_t.extend([t] * len(xs))  # propagate time
        n_segments += 1

    meta = {
        "segments_routed": n_segments,
        "segments_failed_no_path": n_no_path,
        "route_vertices": len(route_x),
        "step": step,
        "max_snap_dist_m": max_snap_dist_m,
    }

    return (
        np.asarray(route_x),
        np.asarray(route_y),
        np.asarray(route_t, dtype="datetime64[ns]"),
        meta,
    )



# -----------------------------
# 2) Sliding-window animation over the matched route vertices
# -----------------------------
def animate_matched_route_trail(
    route_x, route_y, route_t,
    edges_gdf=None,
    out_path="outputs/matched_trail.mp4",
    window_vertices=2000,
    step_vertices=20,
    fps=24,
    dpi=150,
    figsize=(10,10),
    base_edge_alpha=0.18,
    base_edge_lw=0.5,
    trail_lw=2.0,
    trail_alpha=0.95,
    trail_color="black",
    facecolor="white",
    max_frames=None
):
    import os
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    from matplotlib.animation import FuncAnimation, FFMpegWriter, PillowWriter

    os.makedirs(os.path.dirname(out_path) or ".", exist_ok=True)

    if len(route_x) < 2:
        raise ValueError("Route is empty. Try loosening snap distance or reducing step.")

    # Bounds
    if edges_gdf is not None and len(edges_gdf) > 0:
        minx, miny, maxx, maxy = edges_gdf.total_bounds
    else:
        minx, maxx = np.nanmin(route_x), np.nanmax(route_x)
        miny, maxy = np.nanmin(route_y), np.nanmax(route_y)

    pad_x = (maxx - minx) * 0.03 if maxx > minx else 1.0
    pad_y = (maxy - miny) * 0.03 if maxy > miny else 1.0

    fig, ax = plt.subplots(figsize=figsize)
    fig.patch.set_facecolor(facecolor)
    ax.set_facecolor(facecolor)
    ax.set_axis_off()
    ax.set_xlim(minx - pad_x, maxx + pad_x)
    ax.set_ylim(miny - pad_y, maxy + pad_y)

    # Base map drawn ONCE
    if edges_gdf is not None and len(edges_gdf) > 0:
        edges_gdf.plot(ax=ax, linewidth=base_edge_lw, alpha=base_edge_alpha)

    (trail_line,) = ax.plot([], [], lw=trail_lw, alpha=trail_alpha, color=trail_color, animated=True)
    (head_dot,) = ax.plot([], [], marker="o", markersize=4, alpha=0.9, color=trail_color,
                          linestyle="", animated=True)

    # ---- TIME LABEL ----
    time_text = ax.text(
        0.01, 0.99, "",
        transform=ax.transAxes,
        ha="left", va="top",
        fontsize=10,
        bbox=dict(facecolor="white", alpha=0.8, edgecolor="none"),
        animated=True
    )

    n = len(route_x)
    idxs = np.arange(0, n, step_vertices, dtype=int)
    if max_frames is not None:
        idxs = idxs[:max_frames]

    def init():
        trail_line.set_data([], [])
        head_dot.set_data([], [])
        time_text.set_text("")
        return trail_line, head_dot, time_text

    def update(k):
        i = int(idxs[k])
        start = max(0, i - window_vertices + 1)

        xs = route_x[start:i+1]
        ys = route_y[start:i+1]

        # line expects sequences
        trail_line.set_data(xs, ys)

        # dot also expects sequences → wrap scalars in a list
        head_dot.set_data([route_x[i]], [route_y[i]])

        ts = pd.to_datetime(route_t[i]).strftime("%d-%m-%Y / %H:%M:%S")
        time_text.set_text(f"Time: {ts}")

        return trail_line, head_dot, time_text

    anim = FuncAnimation(
        fig, update,
        frames=len(idxs),
        init_func=init,
        blit=True,
        interval=1000 / fps
    )

    ext = os.path.splitext(out_path)[1].lower()
    if ext == ".gif":
        anim.save(out_path, writer=PillowWriter(fps=fps), dpi=dpi)
    else:
        anim.save(out_path, writer=FFMpegWriter(fps=fps, bitrate=1800), dpi=dpi)

    plt.close(fig)
    print(f"Saved: {out_path}")


# -----------------------------
# 3) HOW TO USE (with your existing objects)
# -----------------------------
# Requirements:
# - G: projected graph (ox.project_graph)
# - edges: GeoDataFrame for basemap (same CRS as G)
# - gpath_p: timelinePath points projected to edges.crs and containing 'time'

# Example:
route_x, route_y, route_t, meta = build_matched_route(
    G,
    gpath_p,
    time_col="time",
    step=5,
    max_snap_dist_m=80
)

print(meta)

animate_matched_route_trail(
    route_x, route_y, route_t,
    edges_gdf=edges,
    out_path="outputs/matched_trail_with_time.mp4",
    window_vertices=2000,
    step_vertices=20,
    fps=24,
    dpi=150
)

